<a href="https://colab.research.google.com/github/Anuoluwapo65/pytorch-jax-implementation/blob/main/wandb_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import json
import time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import CIFAR100
from torchvision import transforms

import tqdm.notebook as tqdm


In [ ]:
config = dict(
    num_classes = 100,
    epochs = 156,
    architecture = "CNN",
    batch_size = 128,
    learning_rate = 0.001,
    kernel = [16, 32],
    datasets = "CIFAR100"

)

In [ ]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: abidoyeanuoluwapo47 (abidoyeanuoluwapo47-obafemi-awolowo-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
def model_pipeline(hyperparameters):
   with wandb.init(project = "CIFAR100", config = hyperparameters):
        config = wandb.config

        model, train_loader, val_loader, test_loader, criterion, optimizer =   make(config, device)
        print(model)
        train_model(model, train_loader, val_loader, criterion, optimizer, config)
        test_model(model, test_loader)
   return model

In [ ]:
import torch.utils.data as DataLoader

In [ ]:
def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

print("using this device", device)

using this device cpu


In [ ]:
def make(config, device):
  train_set,val_set, test_set = get_data()
  train_loader,val_loader, test_loader  = make_loader(train_set, val_set, test_set, batch_size = config.batch_size)
  model = ConvNet(config.num_classes, config.kernel).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr = config.learning_rate)
  return model, train_loader, val_loader, test_loader, criterion, optimizer

In [ ]:
train_datasets = CIFAR100(root = "data_path", download = True)
train_datasets

Dataset CIFAR100
    Number of datapoints: 50000
    Root location: data_path
    Split: Train

In [ ]:
img_mean = ((train_datasets.data)/255.0).mean(axis = (0,1,2))
img_std = ((train_datasets.data)/255.0).std(axis = (0,1,2))

In [ ]:
train_transforms = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize(img_mean, img_std),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.RandomResizedCrop(size =32, scale = (0.2, 0.8), ratio = (0.2, 0.8))])
test_transforms = transforms.Compose([transforms.ToTensor(),
                                      transforms.Normalize(img_mean, img_std)])

In [ ]:
from torchvision.datasets import CIFAR100
from torch.utils.data import DataLoader, random_split
import torch

def get_data():
    # Load dataset ONCE
    full_dataset = CIFAR100(
        root="./data",
        train=True,
        download=True,
        transform=train_transforms
    )

    # Compute split sizes
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # Reproducible split
    generator = torch.Generator().manual_seed(42)

    train_set, val_set = random_split(
        full_dataset,
        [train_size, val_size],
        generator=generator
    )

    # Test dataset
    test_set = CIFAR100(
        root="./data",
        train=False,
        download=True,
        transform=test_transforms
    )

    return train_set, val_set, test_set


def make_loader(train_set, val_set, test_set, batch_size):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
# Definition for ConvNet class
class ConvNet(nn.Module):
    def __init__(self, num_classes, kernel_sizes):
        super(ConvNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, kernel_sizes[0], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(kernel_sizes[0], kernel_sizes[1], kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Calculate input features for the fully connected layer
        # Assuming input size is 32x32 after two pooling layers (32/2/2 = 8)
        self.fc = nn.Linear(kernel_sizes[1] * 8 * 8, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        return out

In [ ]:
# Definition for train_model function
def train_model(model, train_loader, val_loader, criterion, optimizer, config):
    model.train()
    print(f"\nStarting training for {config.epochs} epochs...")
    for epoch in tqdm.tqdm(range(config.epochs), desc="Epochs"):
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        # Basic validation loop
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                val_loss += criterion(output, target).item() * data.size(0)
                pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
                correct += pred.eq(target.view_as(pred)).sum().item()
        val_loss /= len(val_loader.dataset)
        accuracy = 100. * correct / len(val_loader.dataset)
        print(f'Epoch {epoch+1}: Train Loss: {loss.item():.4f}, Val Loss: {val_loss:.4f}, Val Acc: {accuracy:.2f}%')
        model.train() # Set back to train mode
    print("Training finished.")

In [ ]:
# Definition for test_model function
def test_model(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item() # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')

In [15]:
model = model_pipeline(config)
model

ConvNet(
  (layer1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=2048, out_features=100, bias=True)
)

Starting training for 156 epochs...


Epochs:   0%|          | 0/156 [00:00<?, ?it/s]

Epoch 1: Train Loss: 3.4829, Val Loss: 3.6766, Val Acc: 15.42%
Epoch 2: Train Loss: 3.6076, Val Loss: 3.5705, Val Acc: 16.71%
Epoch 3: Train Loss: 3.7115, Val Loss: 3.4799, Val Acc: 18.48%
Epoch 4: Train Loss: 3.4104, Val Loss: 3.3898, Val Acc: 20.97%
Epoch 5: Train Loss: 3.0327, Val Loss: 3.3514, Val Acc: 21.46%
Epoch 6: Train Loss: 3.2804, Val Loss: 3.3152, Val Acc: 22.16%
Epoch 7: Train Loss: 3.3174, Val Loss: 3.2614, Val Acc: 22.98%
Epoch 8: Train Loss: 3.1904, Val Loss: 3.2498, Val Acc: 23.54%
Epoch 9: Train Loss: 3.2155, Val Loss: 3.2132, Val Acc: 24.80%
Epoch 10: Train Loss: 3.2082, Val Loss: 3.2208, Val Acc: 23.43%
Epoch 11: Train Loss: 3.0039, Val Loss: 3.2126, Val Acc: 24.41%
Epoch 12: Train Loss: 2.8888, Val Loss: 3.1875, Val Acc: 24.78%
Epoch 13: Train Loss: 3.0495, Val Loss: 3.1798, Val Acc: 24.82%
Epoch 14: Train Loss: 3.0703, Val Loss: 3.1619, Val Acc: 25.20%
Epoch 15: Train Loss: 3.2289, Val Loss: 3.1463, Val Acc: 25.16%
Epoch 16: Train Loss: 3.0414, Val Loss: 3.1393, V

ConvNet(
  (layer1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=2048, out_features=100, bias=True)
)